<div style="background-color: #04D7FD; padding: 20px; text-align: left;">
    <h1 style="color: #000000; font-size: 36px; margin: 0;">Data Processing for RAG with Data Prep Kit (Python)</h1>
    
</div>


## Before Running the notebook

Please complete [setting up python dev environment](./setup-python-dev-env.md)

## Overview

This notebook will process PDF documents as part of RAG pipeline

![](media/rag-overview-2.png)

This notebook will perform steps 1, 2, 3 and 4 in RAG pipeline.

Here are the processing steps:

- **pdf2parquet** : Extract text (in markdown format) from PDF and store them as parquet files
- **Exact Dedup**: Documents with exact content are filtered out
- **Chunk documents**: Split the PDFs into 'meaningful sections' (paragraphs, sentences ..etc)
- **Text encoder**: Convert chunks into vectors using embedding models

## Step-1: Configuration

In [1]:
from my_config import MY_CONFIG

In [2]:
## setup path to utils folder
import sys
sys.path.append('../utils')

## Step-2:  Data

We will use white papers  about LLMs.  

- [Granite Code Models](https://arxiv.org/abs/2405.04324)
- [Attention is all you need](https://arxiv.org/abs/1706.03762)

You can of course substite your own data below

### 2.1 - Download data

In [3]:
import os, sys
import shutil
from file_utils import download_file

shutil.rmtree(MY_CONFIG.INPUT_DATA_DIR, ignore_errors=True)
shutil.os.makedirs(MY_CONFIG.INPUT_DATA_DIR, exist_ok=True)
print ("✅ Cleared input directory")
 
download_file (url = 'https://arxiv.org/pdf/1706.03762', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'attention.pdf' ))
download_file (url = 'https://arxiv.org/pdf/2405.04324', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'granite.pdf' ))
download_file (url = 'https://arxiv.org/pdf/2405.04324', local_file = os.path.join(MY_CONFIG.INPUT_DATA_DIR, 'granite2.pdf' )) # duplicate


✅ Cleared input directory

input/attention.pdf (2.22 MB) downloaded successfully.

input/granite.pdf (1.27 MB) downloaded successfully.

input/granite2.pdf (1.27 MB) downloaded successfully.


### 2.2 - Set input/output path variables for the pipeline

In [4]:
import os, sys
import shutil

if not os.path.exists(MY_CONFIG.INPUT_DATA_DIR ):
    raise Exception (f"❌ Input folder MY_CONFIG.INPUT_DATA_DIR = '{MY_CONFIG.INPUT_DATA_DIR}' not found")

output_parquet_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '01_parquet_out')
output_exact_dedupe_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '02_dedupe_out')
output_chunk_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '03_chunk_out')
output_embeddings_dir = os.path.join (MY_CONFIG.OUTPUT_FOLDER, '04_embeddings_out')

## clear output folder
shutil.rmtree(MY_CONFIG.OUTPUT_FOLDER, ignore_errors=True)
shutil.os.makedirs(MY_CONFIG.OUTPUT_FOLDER, exist_ok=True)

print ("✅ Cleared output directory")

✅ Cleared output directory


## Step-3: docling2parquet -  Convert data from PDF to Parquet

This step is reading the input folder containing all PDF files and ingest them in a parquet table using the [Docling package](https://github.com/DS4SD/docling).
The documents are converted into a JSON format which allows to easily chunk it in the later steps.



### 3.1 - Execute 

In [5]:
%%time 

from dpk_docling2parquet import Docling2Parquet
from dpk_docling2parquet import docling2parquet_contents_types

STAGE = 1
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{MY_CONFIG.INPUT_DATA_DIR}' --> output='{output_parquet_dir}'\n", flush=True)

result = Docling2Parquet(input_folder=MY_CONFIG.INPUT_DATA_DIR,
                    output_folder=output_parquet_dir,
                    data_files_to_use=['.pdf'],
                    docling2parquet_contents_type=docling2parquet_contents_types.MARKDOWN,   # markdown
                    ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-1: Processing input='input' --> output='output/01_parquet_out'



16:02:45 INFO - docling2parquet parameters are : {'batch_size': -1, 'artifacts_path': None, 'contents_type': <docling2parquet_contents_types.MARKDOWN: 'text/markdown'>, 'do_table_structure': True, 'do_ocr': True, 'ocr_engine': <docling2parquet_ocr_engine.EASYOCR: 'easyocr'>, 'bitmap_area_threshold': 0.05, 'pdf_backend': <docling2parquet_pdf_backend.DLPARSE_V2: 'dlparse_v2'>, 'double_precision': 8}
2025-10-03 16:02:45,521 - INFO - docling2parquet parameters are : {'batch_size': -1, 'artifacts_path': None, 'contents_type': <docling2parquet_contents_types.MARKDOWN: 'text/markdown'>, 'do_table_structure': True, 'do_ocr': True, 'ocr_engine': <docling2parquet_ocr_engine.EASYOCR: 'easyocr'>, 'bitmap_area_threshold': 0.05, 'pdf_backend': <docling2parquet_pdf_backend.DLPARSE_V2: 'dlparse_v2'>, 'double_precision': 8}
16:02:45 INFO - pipeline id pipeline_id
2025-10-03 16:02:45,522 - INFO - pipeline id pipeline_id
16:02:45 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'co

✅ Stage:1 completed successfully
CPU times: user 2min 18s, sys: 39.7 s, total: 2min 58s
Wall time: 2min 1s


### 3.2 -  Inspect Generated output

Here we should see one entry per input file processed

In [6]:
from file_utils import read_parquet_files_as_df

output_df = read_parquet_files_as_df(output_parquet_dir)

# print ("Output dimensions (rows x columns)= ", output_df.shape)

output_df.head(5)

## To display certain columns
#parquet_df[['column1', 'column2', 'column3']].head(5)

Successfully read 3 parquet files with 3 total rows


,filename,contents,num_pages,num_tables,num_doc_elements,document_id,document_hash,ext,hash,size,date_acquired,document_convert_time,source_filename
0,granite.pdf,## Granite Code Models: A Family of Open Found...,28,17,485,f593e73e-b6e4-46e4-a849-fc1222166494,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T16:03:51.946152,48.444821,granite.pdf
1,granite2.pdf,## Granite Code Models: A Family of Open Found...,28,17,485,dd254934-b0d0-4076-96a5-e14c1830d3d0,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T16:04:42.039977,50.051021,granite2.pdf
2,attention.pdf,"Provided proper attribution is provided, Googl...",15,4,513,a0e5ae6b-cef1-4fab-b63e-c01df20256b2,2949302674760005271,pdf,214960a61e817387f01087f0b0b323cf1ebd8035fffcab...,48981,2025-10-03T16:03:03.458839,14.119857,attention.pdf


## Step-4: Eliminate Duplicate Documents

We have 2 duplicate documnets here : `granite.pdf` and `granite2.pdf`.

Note how the `hash` for these documents are same.

We are going to perform **de-dupe**

On the content of each document, a SHA256 hash is computed, followed by de-duplication of record having identical hashes.

[Dedupe transform documentation](https://github.com/data-prep-kit/data-prep-kit/blob/dev/transforms/universal/ededup/README.md)

### 4.1 - Execute 

In [7]:
%%time 

from dpk_ededup.transform_python import Ededup

STAGE = 2
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_parquet_dir}' --> output='{output_exact_dedupe_dir}'\n", flush=True)

result = Ededup(input_folder=output_parquet_dir,
    output_folder=output_exact_dedupe_dir,
    ededup_doc_column="contents",
    ededup_doc_id_column="document_id"
    ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-2: Processing input='output/01_parquet_out' --> output='output/02_dedupe_out'



16:05:36 INFO - exact dedup params are {'doc_column': 'contents', 'doc_id_column': 'document_id', 'use_snapshot': False, 'snapshot_directory': None}
2025-10-03 16:05:36,963 - INFO - exact dedup params are {'doc_column': 'contents', 'doc_id_column': 'document_id', 'use_snapshot': False, 'snapshot_directory': None}
16:05:36 INFO - pipeline id pipeline_id
2025-10-03 16:05:36,964 - INFO - pipeline id pipeline_id
16:05:36 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
2025-10-03 16:05:36,965 - INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
16:05:36 INFO - data factory data_ max_files -1, n_sample -1
2025-10-03 16:05:36,966 - INFO - data factory data_ max_files -1, n_sample -1
16:05:36 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
20

✅ Stage:2 completed successfully
CPU times: user 33.4 ms, sys: 14.5 ms, total: 47.9 ms
Wall time: 42.4 ms


### 4.2 - Inspect Generated output

We would see 2 documents: `attention.pdf`  and `granite.pdf`.  The duplicate `granite.pdf` has been filtered out!

In [8]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_parquet_dir)
output_df = read_parquet_files_as_df(output_exact_dedupe_dir)

# print ("Input data dimensions (rows x columns)= ", input_df.shape)
# print ("Output data dimensions (rows x columns)= ", output_df.shape)
print (f"Input files before exact dedupe : {input_df.shape[0]:,}")
print (f"Output files after exact dedupe : {output_df.shape[0]:,}")
print ("Duplicate files removed :  ", (input_df.shape[0] - output_df.shape[0]))

output_df.sample(min(3, output_df.shape[0]))

Successfully read 3 parquet files with 3 total rows
Successfully read 2 parquet files with 2 total rows
Input files before exact dedupe : 3
Output files after exact dedupe : 2
Duplicate files removed :   1


,filename,contents,num_pages,num_tables,num_doc_elements,document_id,document_hash,ext,hash,size,date_acquired,document_convert_time,source_filename
1,attention.pdf,"Provided proper attribution is provided, Googl...",15,4,513,a0e5ae6b-cef1-4fab-b63e-c01df20256b2,2949302674760005271,pdf,214960a61e817387f01087f0b0b323cf1ebd8035fffcab...,48981,2025-10-03T16:03:03.458839,14.119857,attention.pdf
0,granite.pdf,## Granite Code Models: A Family of Open Found...,28,17,485,f593e73e-b6e4-46e4-a849-fc1222166494,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T16:03:51.946152,48.444821,granite.pdf


##  Step-5: Doc chunks

Split the documents in chunks.

[Chunking transform documentation](https://github.com/data-prep-kit/data-prep-kit/blob/dev/transforms/language/doc_chunk/README.md)

**Experiment with chunking size to find the setting that works best for your documents**

### 5.1 - Execute 

In [9]:
%%time

from dpk_doc_chunk.transform_python import DocChunk

STAGE = 3
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_exact_dedupe_dir}' --> output='{output_chunk_dir}'\n", flush=True)

result = DocChunk(input_folder=output_exact_dedupe_dir,
        output_folder=output_chunk_dir,
        doc_chunk_chunking_type= "li_markdown",
        # doc_chunk_chunking_type= "dl_json",
        doc_chunk_chunk_size_tokens = 128,  # default 128
        doc_chunk_chunk_overlap_tokens=30   # default 30
        ).transform()

if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-3: Processing input='output/02_dedupe_out' --> output='output/03_chunk_out'



16:05:42 INFO - doc_chunk parameters are : {'chunking_type': 'li_markdown', 'content_column_name': 'contents', 'doc_id_column_name': 'document_id', 'output_chunk_column_name': 'contents', 'output_source_doc_id_column_name': 'source_document_id', 'output_jsonpath_column_name': 'doc_jsonpath', 'output_pageno_column_name': 'page_number', 'output_bbox_column_name': 'bbox', 'chunk_size_tokens': 128, 'chunk_overlap_tokens': 30, 'dl_min_chunk_len': None}
2025-10-03 16:05:42,332 - INFO - doc_chunk parameters are : {'chunking_type': 'li_markdown', 'content_column_name': 'contents', 'doc_id_column_name': 'document_id', 'output_chunk_column_name': 'contents', 'output_source_doc_id_column_name': 'source_document_id', 'output_jsonpath_column_name': 'doc_jsonpath', 'output_pageno_column_name': 'page_number', 'output_bbox_column_name': 'bbox', 'chunk_size_tokens': 128, 'chunk_overlap_tokens': 30, 'dl_min_chunk_len': None}
16:05:42 INFO - pipeline id pipeline_id
2025-10-03 16:05:42,333 - INFO - pipeli

✅ Stage:3 completed successfully
CPU times: user 485 ms, sys: 132 ms, total: 617 ms
Wall time: 737 ms


### 5.2 - Inspect Generated output

We would see documents are split into many chunks

In [10]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_exact_dedupe_dir)  ## for debug purposes
output_df = read_parquet_files_as_df(output_chunk_dir)

print (f"Files processed : {input_df.shape[0]:,}")
print (f"Chunks created : {output_df.shape[0]:,}")

# print ("Input data dimensions (rows x columns)= ", input_df.shape)
# print ("Output data dimensions (rows x columns)= ", output_df.shape)

output_df.sample(min(3, output_df.shape[0]))

Successfully read 2 parquet files with 2 total rows
Successfully read 2 parquet files with 61 total rows
Files processed : 2
Chunks created : 61


,filename,num_pages,num_tables,num_doc_elements,document_hash,ext,hash,size,date_acquired,document_convert_time,source_filename,source_document_id,contents,document_id
60,attention.pdf,15,4,513,2949302674760005271,pdf,214960a61e817387f01087f0b0b323cf1ebd8035fffcab...,48981,2025-10-03T16:03:03.458839,14.119857,attention.pdf,a0e5ae6b-cef1-4fab-b63e-c01df20256b2,## Attention Visualizations Input-Input Layer5...,8d0f64fb29fb663c3002a06889a79cea125f2e1f86c61a...
56,attention.pdf,15,4,513,2949302674760005271,pdf,214960a61e817387f01087f0b0b323cf1ebd8035fffcab...,48981,2025-10-03T16:03:03.458839,14.119857,attention.pdf,a0e5ae6b-cef1-4fab-b63e-c01df20256b2,## 6.2 Model Variations\n\nTo evaluate the imp...,af3a7968218bbd719168314e905a7f2e60b1b6ce916192...
34,attention.pdf,15,4,513,2949302674760005271,pdf,214960a61e817387f01087f0b0b323cf1ebd8035fffcab...,48981,2025-10-03T16:03:03.458839,14.119857,attention.pdf,a0e5ae6b-cef1-4fab-b63e-c01df20256b2,## Attention Is All You Need\n\nAshish Vaswani...,573d1e20f1f71713f4f318a4abdffbb5109437d4fc02a2...


## Step-6:   Calculate Embeddings for Chunks

we will calculate embeddings for each chunk using an open source embedding model

[Embeddings / Text Encoder documentation](https://github.com/data-prep-kit/data-prep-kit/blob/dev/transforms/language/text_encoder/README.md)

### 6.1 - Execute

In [13]:
%%time 

from dpk_text_encoder.transform_python import TextEncoder

STAGE  = 4
print (f"🏃🏼 STAGE-{STAGE}: Processing input='{output_chunk_dir}' --> output='{output_embeddings_dir}'\n", flush=True)


result = TextEncoder(input_folder= output_chunk_dir, 
               output_folder= output_embeddings_dir, 
               text_encoder_model_name = MY_CONFIG.EMBEDDING_MODEL
               ).transform()
if result == 0:
    print (f"✅ Stage:{STAGE} completed successfully")
else:
    raise Exception (f"❌ Stage:{STAGE}  failed")

🏃🏼 STAGE-4: Processing input='output/03_chunk_out' --> output='output/04_embeddings_out'



16:05:58 INFO - text_encoder parameters are : {'content_column_name': 'contents', 'output_embeddings_column_name': 'embeddings', 'model_name': 'ibm-granite/granite-embedding-30m-english'}
2025-10-03 16:05:58,466 - INFO - text_encoder parameters are : {'content_column_name': 'contents', 'output_embeddings_column_name': 'embeddings', 'model_name': 'ibm-granite/granite-embedding-30m-english'}
16:05:58 INFO - pipeline id pipeline_id
2025-10-03 16:05:58,467 - INFO - pipeline id pipeline_id
16:05:58 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
2025-10-03 16:05:58,468 - INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
16:05:58 INFO - data factory data_ max_files -1, n_sample -1
2025-10-03 16:05:58,469 - INFO - data factory data_ max_files -1, n_sample -1
16:05:58 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, ran

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

16:06:01 INFO - Completed 2 files (100.0%) in 0.018 min
2025-10-03 16:06:01,447 - INFO - Completed 2 files (100.0%) in 0.018 min
16:06:01 INFO - Done processing 2 files, waiting for flush() completion.
2025-10-03 16:06:01,447 - INFO - Done processing 2 files, waiting for flush() completion.
16:06:01 INFO - done flushing in 0.0 sec
2025-10-03 16:06:01,448 - INFO - done flushing in 0.0 sec
16:06:01 INFO - Completed execution in 0.05 min, execution result 0
2025-10-03 16:06:01,449 - INFO - Completed execution in 0.05 min, execution result 0


✅ Stage:4 completed successfully
CPU times: user 977 ms, sys: 355 ms, total: 1.33 s
Wall time: 2.99 s


### 6.2 - Inspect Generated output

In [14]:
from file_utils import read_parquet_files_as_df

input_df = read_parquet_files_as_df(output_chunk_dir)
output_df = read_parquet_files_as_df(output_embeddings_dir)

print ("Input data dimensions (rows x columns)= ", input_df.shape)
print ("Output data dimensions (rows x columns)= ", output_df.shape)

output_df.sample(min(3, output_df.shape[0]))

Successfully read 2 parquet files with 61 total rows
Successfully read 2 parquet files with 61 total rows
Input data dimensions (rows x columns)=  (61, 14)
Output data dimensions (rows x columns)=  (61, 15)


,filename,num_pages,num_tables,num_doc_elements,document_hash,ext,hash,size,date_acquired,document_convert_time,source_filename,source_document_id,contents,document_id,embeddings
9,granite.pdf,28,17,485,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T16:03:51.946152,48.444821,granite.pdf,f593e73e-b6e4-46e4-a849-fc1222166494,"## 4 Pretraining\n\nIn this section, we provid...",b1dfa24a41fa6d6040704db6bc96c47ac1a09da191af05...,"[-0.0498102, -0.06920969, -0.0095467325, 0.026..."
4,granite.pdf,28,17,485,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T16:03:51.946152,48.444821,granite.pdf,f593e73e-b6e4-46e4-a849-fc1222166494,## 2.1 Data Crawling and Filtering\n\nThe pret...,64f536c6e279db81ccda1ba14e35ca553e4250de346176...,"[0.049523477, -0.015281111, 0.0320079, 0.04259..."
16,granite.pdf,28,17,485,3127757990743433032,pdf,58342470e7d666dca0be87a15fb0552f949a5632606fe1...,121131,2025-10-03T16:03:51.946152,48.444821,granite.pdf,f593e73e-b6e4-46e4-a849-fc1222166494,## 6.1 Code Generation,719ad78af25dc40f13c6d3273faa31d4c5f48b5fbba1fd...,"[-0.005764263, 0.014997978, 0.03899995, 0.0295..."


## Step-7: Copy output to final output dir

In [15]:
import shutil

shutil.rmtree(MY_CONFIG.OUTPUT_FOLDER_FINAL, ignore_errors=True)
shutil.copytree(src=output_embeddings_dir, dst=MY_CONFIG.OUTPUT_FOLDER_FINAL)

print (f"✅ Copied output from '{output_embeddings_dir}' --> '{MY_CONFIG.OUTPUT_FOLDER_FINAL}'")

✅ Copied output from 'output/04_embeddings_out' --> 'output/output_final'
